In [7]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 16384  # Choose any! We auto support RoPE Scaling internally!
dtype = None  # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True  # Use 4bit quantization to reduce memory usage. Can be False.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-unsloth-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit, device_map="cuda"
)
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference
print("Load Finished")

from util.pdf2md import md_paper_pipeline, md_book_pipeline, md_book_pipeline_seperate
import re
import os
import ast
import json
from tqdm.autonotebook import tqdm

==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 5090. Num GPUs = 1. Max memory: 31.842 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.30. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 3/3 [00:03<00:00,  1.18s/it]


Load Finished


In [8]:
chat_history = []
temperature = 0.8  # Controls randomness in generation (higher = more random)
top_p = 0.9
top_k = 50
repetition_penalty = 1.5
system_prompt = (
    "You are a scanning probe microscopy (SPM) specialist tasked with helping me create an educational SPM knowledge dataset. "
    "I will provide you with paragraphs from an SPM research paper. Based on each paragraph, ask questions that help learners understand the scientific, technical content and learn new scientific or technical knowledge from the content. "
    "The questions should focus on extracting knowledge related to scientific principles, device mechanisms, experimental algorithms, and methodologies."
    "In other words, the questions you ask must be centered around scientific knowledge, instrument principles, experimental algorithms and methods that help people gain new understanding. "
    "The questions must be phrased in a general way so that they are understandable without reading the original article. "
    "If people have knowledgve, they can answer without reading this paper, so do not reference the text directly with expressions such as *mentioned in the text*, *outlined in the article*, *in Figure 1*, etc. "
    "Generate a total of 50 questions. "
    "The questions should output in a list format in python. following: question_list = []"
)

system_prompt_token_count = len(tokenizer.encode(system_prompt))
max_new_tokens_count = 2500
max_input_token_count = max_seq_length - system_prompt_token_count - max_new_tokens_count - 50

def generate(input_text):
    user_msg = {"role": "user", "content": input_text}

    prompt_text = [
        {"role": "system", "content": system_prompt},
    ]
    prompt_text.append(user_msg)

    inputs = tokenizer.apply_chat_template(
        prompt_text,
        tokenize=True,
        add_generation_prompt=True,  # Must add for generation
        return_tensors="pt",
    ).to("cuda")

    print(f"input token count is {len(inputs[0])}")

    inputs_text = tokenizer.decode(inputs[0])
    # print(inputs_text)
    outputs = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens_count, use_cache=True,
                             temperature=temperature, min_p=0.1)

    outputs = tokenizer.decode(outputs[0]).split(inputs_text)[-1]

    chat_history.append((user_msg, {"role": "assistant", "content": outputs}))
    return outputs






### Extract question from book

Long chapters and can divide by chapters.
Feed to LLM for each chapter.

In [9]:
def write_book_datasetv2(path):
    basename = os.path.basename(path)

    text_list = []
    file_dict = {"type": "book", "answered": False}

    chapter_list = md_book_pipeline(path)
    for i, chapter in enumerate(tqdm(chapter_list[1:])):
        text = chapter["content"].strip()
        
        tokens = tokenizer.encode(text)
        if len(tokens) > max_input_token_count:
            print(f"chapter {i + 2} is too long ({len(tokens)}), over {max_input_token_count} words will be cut.")
            text = tokenizer.decode(tokens[:max_input_token_count])

        outputs = generate(text)
        regions = re.findall(r'```python\s*(.*?)```', outputs, flags=re.DOTALL)[0]
        lst = ast.literal_eval("".join(regions.split("=")[1:]))

        text_list.append(text)
        file_dict[str(i)] = lst

    file_dict["text"] = text_list
    output_dir = "outputs/"
    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}{basename}.json", "w") as json_file:
        json.dump(file_dict, json_file, indent=4)
        

def write_book_dataset_seperate(path_folder):
    basename = os.path.basename(os.path.normpath(path_folder))

    text_list = []
    file_dict = {"type": "book", "answered": False}

    chapter_list = md_book_pipeline_seperate(path_folder)
    for i, chapter in enumerate(tqdm(chapter_list)):
        text = chapter["content"].strip()
        
        tokens = tokenizer.encode(text)
        if len(tokens) > max_input_token_count:
            print(f"chapter {i + 2} is too long ({len(tokens)}), over {max_input_token_count} words will be cut.")
            text = tokenizer.decode(tokens[:max_input_token_count])

        outputs = generate(text)
        regions = re.findall(r'```python\s*(.*?)```', outputs, flags=re.DOTALL)[0]
        lst = ast.literal_eval("".join(regions.split("=")[1:]))

        text_list.append(text)
        file_dict[str(i)] = lst

    file_dict["text"] = text_list
    output_dir = "outputs/"
    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}{basename}.json", "w") as json_file:
        json.dump(file_dict, json_file, indent=4)



### Extract question from paper

Simplly feed all text to LLM.

In [10]:
def write_journal_dataset(path):
    basename = os.path.basename(path)
    text = md_paper_pipeline(path)
    outputs = generate(text)
    regions = re.findall(r'```python\s*(.*?)```', outputs, flags=re.DOTALL)[0]
    # print("".join(regions.split("=")[1:]))
    lst = ast.literal_eval("".join(regions.split("=")[1:]))

    file_dict = {"questions": lst, "text": text, "type": "journal", "answered": False}
    output_dir = "outputs/"
    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}{basename}.json", "w") as json_file:
        json.dump(file_dict, json_file, indent=4)


### Extract question from long paper or book

append chapter to text limit (12000 tokens) and then feed to LLM.

In [11]:
def write_book_datasetv3(path):
    def get_token_count(text):
        return len(tokenizer.encode(text))
    
    basename = os.path.basename(path)

    text_list = []
    file_dict = {"type": "book", "answered": False}

    chapter_list = md_book_pipeline(path)
    for chapter in chapter_list:
        chapter["token_count"] = get_token_count(chapter["content"].strip())
    
        
    add_text_list = []
    iteration = 0
    for i, chapter in enumerate(tqdm(chapter_list)):
        if i in add_text_list:
            continue
        
        
        text = chapter["content"].strip()
        text_tokens = chapter["token_count"] 
        
        for k in range(i+1, len(chapter_list)):
            if chapter_list[k]["token_count"] + text_tokens < max_input_token_count:
                add_text_list.append(k)
                text_tokens += chapter_list[k]["token_count"]
                text += chapter_list[k]["content"].strip()
                print(f"mix chapter {k+1} to chapter {i+1}")
            
        tokens = tokenizer.encode(text)
        if len(tokens) > max_input_token_count:
            print(f"chapter {i + 1} is too long ({len(tokens)}), over {max_input_token_count} words will be cut.")
            text = tokenizer.decode(tokens[:max_input_token_count])

        outputs = generate(text)
        regions = re.findall(r'```python\s*(.*?)```', outputs, flags=re.DOTALL)[0]
        lst = ast.literal_eval("".join(regions.split("=")[1:]))

        text_list.append(text)
        file_dict[str(iteration)] = lst
        iteration += 1


    file_dict["text"] = text_list
    output_dir = "outputs/"
    os.makedirs(output_dir, exist_ok=True)
    with open(f"{output_dir}{basename}.json", "w") as json_file:
        json.dump(file_dict, json_file, indent=4)


In [12]:
write_journal_dataset(
    "../spm_paper/paper/Small Methods - 2024 - Diao - AI‐Equipped Scanning Probe Microscopy for Autonomous Site‐Specific Atomic‐Level.pdf")

Processing ../spm_paper/paper/Small Methods - 2024 - Diao - AI‐Equipped Scanning Probe Microscopy for Autonomous Site‐Specific Atomic‐Level.pdf...


100%|██████████| 10/10 [00:00<00:00, 20.93it/s]


input token count is 10795
